In [ ]:
'''
GRAPPA reconstruction with an iterative algorithm from CIL: illustrates
the use of AcquisitionModel in CIL optimisation 

Usage:
  grappa_and_cil.py [--help | options]

Options:
  -f <file>, --file=<file>    raw data file
                              [default: simulated_MR_2D_cartesian_Grappa2.h5]
  -p <path>, --path=<path>    path to data files, defaults to data/examples/MR
                              subfolder of SIRF root folder
'''

## CCP PETMR Synergistic Image Reconstruction Framework (SIRF)
## Copyright 2015 - 2019 Rutherford Appleton Laboratory STFC.
## Copyright 2015 - 2019 University College London.
##
## This is software developed for the Collaborative Computational
## Project in Positron Emission Tomography and Magnetic Resonance imaging
## (http://www.ccppetmr.ac.uk/).
##
## Licensed under the Apache License, Version 2.0 (the "License");
##   you may not use this file except in compliance with the License.
##   You may obtain a copy of the License at
##       http://www.apache.org/licenses/LICENSE-2.0
##   Unless required by applicable law or agreed to in writing, software
##   distributed under the License is distributed on an "AS IS" BASIS,
##   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
##   See the License for the specific language governing permissions and
##   limitations under the License.
    

import sirf
from sirf.Utilities import existing_filepath
from sirf.Utilities import error
from sirf.Utilities import show_3D_array
from sirf.Gadgetron import examples_data_path
from sirf.Gadgetron import AcquisitionData, ImageData
from sirf.Gadgetron import AcquisitionModel
from sirf.Gadgetron import AcquisitionDataProcessor
from sirf.Gadgetron import CartesianGRAPPAReconstructor, FullySampledReconstructor
from sirf.Gadgetron import CoilSensitivityData
from sirf.Gadgetron import preprocess_acquisition_data

from cil.optimisation.functions import LeastSquares
from cil.optimisation.functions import ZeroFunction
from cil.optimisation.algorithms import FISTA, CGLS, GD
from cil.plugins.ccpi_regularisation.functions import FGP_TV#, TGV, LLT_ROF, Diff4th
from cil.framework import DataContainer as cilDataContainer
from cil.optimisation.operators import LinearOperator

import numpy
import time
# %matplotlib notebook
import matplotlib.pyplot as plt
import os


In [ ]:
# process command-line options
fname_ai = '1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL_mod.h5'
fname_full = '1meas_MID00614_FID129152_CONVENTIONAL_RECON_SEQD_GF2_AX_RL_mod.h5'

data_path = '/home/jovyan/work/data/h5'

input_file = os.path.join(data_path, fname_full)
print (input_file)
# acquisition data will be read from an HDF file input_data
acq_data_full = AcquisitionData(input_file)

In [ ]:
acq_data = preprocess_acquisition_data(acq_data_full)

In [ ]:
recon = FullySampledReconstructor()
recon.set_input(acq_data)
recon.process()


In [ ]:
from cil.utilities.display import show2D
from cil.utilities.jupyter import islicer
import numpy as np

full_recon = np.abs(recon.get_output().asarray(), dtype=np.float32)


In [ ]:
show2D(full_recon)

In [7]:
print(type(full_recon), full_recon.dtype)

<class 'numpy.ndarray'> float32


In [102]:
# downsampled data
input_file = os.path.join(data_path, fname_ai)
acq_data_ai = AcquisitionData(input_file)
acq_data_ai = preprocess_acquisition_data(acq_data_ai)

reading from /home/jovyan/work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL_mod.h5 using ignore mask 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 

Started reading acquisitions from /home/jovyan/work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL_mod.h5
0%..10%..20%..30%..40%..50%..60%..70%..80%..90%..100%..
Finished reading acquisitions from /home/jovyan/work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL_mod.h5
ignoring acquisition 0
Message received with ID: 5
Input stream has terminated


In [103]:
csm = CoilSensitivityData()
csm.smoothness = 100
csm.calculate(acq_data_ai)

In [104]:
E = AcquisitionModel(acqs=acq_data_ai, imgs=csm)
E.set_coil_sensitivity_maps(csm)
x_inverse = E.inverse(acq_data_ai)

In [127]:
# We set up our AcquisitionModel
E = AcquisitionModel(acqs=acq_data_ai, imgs=x_inverse)
E.set_coil_sensitivity_maps(csm)


# Use the result of the inverse as our starting point
x_init = x_inverse.clone()

# Define our objective/loss function as least squares between Ex and y
f = LeastSquares(E, acq_data_ai, c=1)

G = ZeroFunction()

# Set up FISTA
fista = FISTA(initial=x_init.fill(0.0), f=f, g=G)
fista.update_objective_interval = 5


# Run FISTA for least squares
fista.run(10)

  0%|          | 0/10 [00:00<?, ?it/s]

In [128]:
ai_recon = np.abs(fista.solution.asarray(), dtype=np.float32)

In [129]:
# normalise results
def normalise(data):
    return (data - data.min())/(data.max()-data.min())


fn_recon = normalise(full_recon)
ain_recon = normalise(ai_recon)

In [ ]:
# ER rotate images for powerpoint
fn_recon_rotated = np.rot90(fn_recon, axes=(2, 1))
ain_recon_rotated = np.rot90(ain_recon, axes=(2, 1))

show2D([fn_recon_rotated, ain_recon_rotated], slice_list=(0, 6), fix_range=(np.percentile(fn_recon, 5), np.percentile(fn_recon, 95)))

In [ ]:
from cil.optimisation.functions import L1Sparsity, FunctionOfAbs
from cil.optimisation.operators import WaveletOperator
from cil.plugins.ccpi_regularisation.functions import FGP_TV

TV = FGP_TV(alpha = 0.3)


In [ ]:

algo_tv = FISTA(initial=x_init.fill(0.0), f=f, g=TV, update_objective_interval=1)
print("algo configured")
algo_tv.run(70)

In [ ]:
# algo_tv.run(60)

In [ ]:
tv_recon = np.abs(algo_tv.solution.asarray(), dtype=np.float32)
tv_recon = normalise(tv_recon)
show2D([el[:,230:330, 190:290] for el in [fn_recon, tv_recon]],
       title=['Fully sampled', 'LS+TV'],
       slice_list=(0, 5), 
       fix_range=(np.percentile(fn_recon, 5), np.percentile(fn_recon, 99)),
      num_cols=2)

show2D([el[:,100:400, 90:390] for el in [fn_recon, tv_recon]],
       title=['Fully sampled', 'LS+TV'],
       slice_list=(0, 5), 
       fix_range=(np.percentile(fn_recon, 5), np.percentile(fn_recon, 99)),
      num_cols=2)

In [ ]:
# ER rotate images for powerpoint
tv_recon_rotated = np.rot90(tv_recon, axes=(2, 1))
fn_recon_rotated = np.rot90(fn_recon, axes=(2, 1))

show2D([el[:,230:330, 190:290] for el in [fn_recon_rotated, tv_recon_rotated]],
       title=['Fully sampled', 'LS+TV'],
       slice_list=(0, 5), 
       fix_range=(np.percentile(fn_recon, 5), np.percentile(fn_recon, 99)),
      num_cols=2)

In [64]:
w = WaveletOperator(x_init, wname="bior4.4", level=1)
R_W = FunctionOfAbs(
    L1Sparsity(w)
)

/opt/conda/lib/python3.12/site-packages/cil/optimisation/functions/L1Sparsity.py:54: UserWarning: Invalid operator: `<cil.optimisation.operators.WaveletOperator.WaveletOperator object at 0x7f4fc038a3c0>`. L1Sparsity is properly defined only for orthogonal operators!
  warnings.warn(


In [68]:

x_init.geometry = x_init
algo_w = FISTA(x_init, f=f, g=R_W)
algo_w.run(10)


AttributeError: 'ImageData' object has no attribute 'geometry'

In [ ]:
np.save??